# Data Cleaning
#### Preparing the loaded datasets for analysis (H1B, PPP, and the salaries dataset)
**Input:** Raw CSVs from data/raw_data  
**Output:** cleaned CSVs to data/cleaned_data  
**Author: Kunal Verma**

In [1]:
import numpy as np
import pandas as pd

In [2]:
df_h1b = pd.read_csv('../data/raw_data/h1b_raw.csv')
df_levels = pd.read_csv('../data/raw_data/levels_raw.csv')
df_ppp = pd.read_csv('../data/raw_data/ppp_raw.csv')

print("Shape of h1b:", df_h1b.shape)
print("Shape of levels:", df_levels.shape)
print("Shape of ppp:", df_ppp.shape)

Shape of h1b: (210387, 8)
Shape of levels: (16494, 5)
Shape of ppp: (266, 3)


In [3]:
# checking the total null values in the datasets

print('-------- H1B --------')
print(df_h1b.isnull().sum())

print('\n-------- LEVELS.FYI --------')
print(df_levels.isnull().sum())

print('\n-------- PPP --------')
print(df_ppp.isnull().sum())

# isnull().sum(), this function tells us the total number of missing values in each column.

-------- H1B --------
CASE_STATUS               0
JOB_TITLE                 0
EMPLOYER_NAME             0
EMPLOYER_CITY             0
EMPLOYER_STATE           41
EMPLOYER_COUNTRY          0
WAGE_RATE_OF_PAY_FROM     0
WAGE_UNIT_OF_PAY          0
dtype: int64

-------- LEVELS.FYI --------
job_title             0
experience_level      0
salary_in_usd         0
employee_residence    0
employment_type       0
dtype: int64

-------- PPP --------
country          0
country_code     0
ppp_2024        77
dtype: int64


In [4]:
# Cleaning H1B dataset first
# 1) keeping only the certified cases 2) dropping the null 41 employer state 3) converting hourly wages to annual 4) lowercase column names.

df_h1b = df_h1b[df_h1b['CASE_STATUS'] == 'Certified']
df_h1b = df_h1b.dropna(subset = ['EMPLOYER_STATE'])
df_h1b['WAGE_RATE_OF_PAY_FROM'] = df_h1b.apply(
    lambda x : x['WAGE_RATE_OF_PAY_FROM']* 2080 if x['WAGE_UNIT_OF_PAY'] == 'Hour'
    else x['WAGE_RATE_OF_PAY_FROM'] , axis = 1
)

df_h1b = df_h1b.drop(columns=['WAGE_UNIT_OF_PAY', 'CASE_STATUS'])
df_h1b.columns = df_h1b.columns.str.lower()

print(df_h1b.shape)
df_h1b.head(10)

(190122, 6)


,job_title,employer_name,employer_city,employer_state,employer_country,wage_rate_of_pay_from
177,Network Architect,University of Montana,Missoula,MT,UNITED STATES OF AMERICA,96000.00
178,Applied AI Engineer,Google LLC,Mountain View,CA,UNITED STATES OF AMERICA,168000.00
179,Senior Database Administrator,Symphony Corporation,Madison,WI,UNITED STATES OF AMERICA,128000.00
180,Senior Developer,HCL AMERICA INC,Santa Clara,CA,UNITED STATES OF AMERICA,103653.00
181,Software Developer,TATA CONSULTANCY SERVICES LIMITED,ROCKVILLE,MD,UNITED STATES OF AMERICA,134971.00
182,Senior Enterprise Data Governance Analyst,Zoox Inc.,Foster City,CA,UNITED STATES OF AMERICA,164403.00
183,Assistant Professor,Grambling State University,Grambling,LA,UNITED STATES OF AMERICA,64358.44
184,"Manager III, Software Development",Amazon.com Services LLC,Seattle,WA,UNITED STATES OF AMERICA,217244.00
185,Product Engineer,KLA Corporation,Milpitas,CA,UNITED STATES OF AMERICA,126515.00
186,Software Engineering – Java J2EE - Advanced,Silicon Software Technologies Inc,Plano,TX,UNITED STATES OF AMERICA,134971.00


In [5]:
# Cleaning the PPP dataset now
# dropping the nulls

df_ppp = df_ppp.dropna(subset = ['ppp_2024'])
print(df_ppp.shape)
df_ppp.head(10)

(189, 3)


,country,country_code,ppp_2024
0,Aruba,ABW,1.395922
4,Angola,AGO,265.840925
5,Albania,ALB,39.667384
6,Andorra,AND,0.607839
8,United Arab Emirates,ARE,2.330334
9,Argentina,ARG,419.901040
10,Armenia,ARM,147.231105
12,Antigua and Barbuda,ATG,1.903920
13,Australia,AUS,1.366527
14,Austria,AUT,0.710451


In [6]:
# Cleaning the levels dataset
# 1) Filtering only the full time exployees 2) Keeping our focus countries only: ['IN', 'US', 'GB', 'DE', 'NL']

df_levels = df_levels[df_levels['employment_type'] == 'FT']
target_countries = ['IN', 'US', 'GB', 'DE', 'NL']
df_levels = df_levels[df_levels['employee_residence'].isin(target_countries)]
df_levels = df_levels.drop(columns = ['employment_type'])

print(df_levels.shape)
df_levels.head(10)

(15273, 4)


,job_title,experience_level,salary_in_usd,employee_residence
4,Data Science Manager,MI,240000,US
5,Data Science Manager,MI,180000,US
6,Business Intelligence Engineer,SE,202800,US
7,Business Intelligence Engineer,SE,115000,US
8,Data Architect,SE,100000,GB
9,Data Architect,SE,56250,GB
12,Data Analyst,MI,112300,US
13,Data Analyst,MI,75100,US
14,Business Intelligence,MI,178200,US
15,Business Intelligence,MI,87000,US


In [7]:
#PPP adjustment
# levels and ppp merged here
df_levels = df_levels.merge(
    df_ppp[['country_code', 'ppp_2024']],
    left_on = 'employee_residence',
    right_on = 'country_code',
    how = 'left'
)
df_levels = df_levels.drop(columns = ['country_code'])
print(df_levels.shape)
df_levels.head(10)

(15273, 5)


,job_title,experience_level,salary_in_usd,employee_residence,ppp_2024
0,Data Science Manager,MI,240000,US,NaN
1,Data Science Manager,MI,180000,US,NaN
2,Business Intelligence Engineer,SE,202800,US,NaN
3,Business Intelligence Engineer,SE,115000,US,NaN
4,Data Architect,SE,100000,GB,NaN
5,Data Architect,SE,56250,GB,NaN
6,Data Analyst,MI,112300,US,NaN
7,Data Analyst,MI,75100,US,NaN
8,Business Intelligence,MI,178200,US,NaN
9,Business Intelligence,MI,87000,US,NaN


In [8]:
country_code_map = {
    'US': 'USA',
    'IN': 'IND',
    'GB': 'GBR',
    'DE': 'DEU',
    'NL': 'NLD'
}

df_levels['country_code_3'] = df_levels['employee_residence'].map(country_code_map)

df_levels = df_levels.merge(
    df_ppp[['country_code', 'ppp_2024']],
    left_on='country_code_3',
    right_on='country_code',
    how='left'
)

# just print columns before dropping anything
print(df_levels.columns.tolist())

['job_title', 'experience_level', 'salary_in_usd', 'employee_residence', 'ppp_2024_x', 'country_code_3', 'country_code', 'ppp_2024_y']


In [9]:
# reload levels from the original raw file to start completely fresh
df_levels = pd.read_csv('../data/raw_data/levels_raw.csv')

# filter full time and target countries
df_levels = df_levels[df_levels['employment_type'] == 'FT']
target_countries = ['US', 'IN', 'GB', 'DE', 'NL']
df_levels = df_levels[df_levels['employee_residence'].isin(target_countries)]
df_levels = df_levels.drop(columns=['employment_type'])

# now merge ppp cleanly
country_code_map = {'US':'USA','IN':'IND','GB':'GBR','DE':'DEU','NL':'NLD'}
df_levels['country_code_3'] = df_levels['employee_residence'].map(country_code_map)
df_levels = df_levels.merge(df_ppp[['country_code','ppp_2024']], left_on='country_code_3', right_on='country_code', how='left')
df_levels = df_levels.drop(columns=['country_code_3', 'country_code'])

print(df_levels.columns.tolist())
print(df_levels.shape)
df_levels.head(5)

['job_title', 'experience_level', 'salary_in_usd', 'employee_residence', 'ppp_2024']
(15273, 5)


,job_title,experience_level,salary_in_usd,employee_residence,ppp_2024
0,Data Science Manager,MI,240000,US,1.000000
1,Data Science Manager,MI,180000,US,1.000000
2,Business Intelligence Engineer,SE,202800,US,1.000000
3,Business Intelligence Engineer,SE,115000,US,1.000000
4,Data Architect,SE,100000,GB,0.664153


In [10]:
df_levels['salary_ppp_adjusted'] = df_levels['salary_in_usd']/df_levels['ppp_2024']

In [11]:
# Removal of all the salary outliers

df_levels = df_levels[(df_levels['salary_in_usd'] >= 10000) & (df_levels['salary_in_usd'] <= 800000)]
df_h1b = df_h1b[(df_h1b['wage_rate_of_pay_from'] >= 10000) & (df_h1b['wage_rate_of_pay_from'] <= 800000)]
print(df_levels.shape)
print(df_h1b.shape)

(15273, 6)
(189626, 6)


In [12]:
df_h1b.to_csv('../data/cleaned_data/h1b_cleaned.csv', index = False)
df_levels.to_csv('../data/cleaned_data/levels_cleaned.csv', index=False)
df_ppp.to_csv('../data/cleaned_data/ppp_cleaned.csv', index=False)

print("All the cleaned datasets have been saved!")

All the cleaned datasets have been saved!


In [13]:
print(df_h1b['wage_rate_of_pay_from'].describe())

count    189626.000000
mean     141549.186734
std       60433.215723
min       10000.000000
25%      100905.750000
50%      131040.000000
75%      169612.925000
max      800000.000000
Name: wage_rate_of_pay_from, dtype: float64
